In [1]:
import numpy as np
import matplotlib.pyplot as plt
import einops
import torch
import torchvision.transforms as transforms
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from ipywidgets import interact, FloatSlider, Checkbox
# import torch_utils   # utilitaire local (sélection GPU)
import seaborn as sns
import mediapy as mp
import joblib
import os
import random
import cv2
import json

In [2]:
def which_device() -> str:
    """
    Detects the best available device for PyTorch-based inference (CUDA, MPS, XLA/TPU, or CPU).

    Returns
    -------
    str
        The best available device, one of: 'cuda', 'mps', 'xla', or 'cpu'.
    """

    # 1. CUDA (NVIDIA GPUs)
    if torch.cuda.is_available():
        print(f"✅ Using CUDA GPU: {torch.cuda.get_device_name(0)}")
        return "cuda"

    # 2. MPS (Apple Silicon GPUs)
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("✅ Using Apple Silicon GPU (MPS)")
        return "mps"

    # 3. TPU (XLA - PyTorch/XLA)
    try:
        import torch_xla.core.xla_model as xm
        dev = xm.xla_device()
        print(f"✅ Using TPU: {dev}")
        return "xla"
    except ImportError:
        pass  # torch_xla not installed or no TPU available

    # 4. CPU fallback
    print("⚠️ Using CPU")
    return "cpu"

In [3]:
# Depending on the compute power available, we can use more or less tokens and do more of less PCA components.
N_PATCHES = 100          
N_PCA_COMPONENTS = 25
IMG_SIZE = N_PATCHES * 14 
PATCH_SIZE=14


In [4]:
def extract_features_from_one_image(image_path, model, device):
    """This function extracts the features of each patches from a singular image, and outputs them in a tensor of shape (N_PATCHE*N_PATCHES, Dimension_of_DINO_embedding)""" 
    img1= image_path
   
    
    # ImageNet transformation pipeline, used because DINOv2 was trained on such images. Images are resized to squares of N_PATCH*N_PATCH size, and RGB values normalized.
    prep_images = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        ),
    ])
    im1 = Image.open(img1).convert('RGB')
    tim1 = prep_images(im1).to(device)
    with torch.no_grad():
        ret1 = model.forward_features(tim1[None]) #Get DINOv2 features
    tok1 = ret1['x_norm_patchtokens'].squeeze() #Extract the patch tokens only from all the features
    return tok1

In [5]:
def combine_features(liste_tokens):
    """ Combines the patch tokens from several images"""
    tok = np.vstack([i.cpu().numpy() for i in liste_tokens]) 
    return tok

In [6]:
def fit_pca2(tok_fit):
    """Fits a PCA on the combination of image features created in the combine_feature function, and saves it as pca_reference.pk1"""
    pca = PCA(n_components=N_PCA_COMPONENTS)
    pca.fit(tok_fit)
    joblib.dump(pca, "pca_reference.pkl")
    print("PCA_fit saved as pca_reference.pkl")

In [7]:
def fit_pca(images_folder):
    """get the features of all images from a folder, then combines them into a tok and fit a pca on it"""
    device = which_device()
    print(f"Device utilisé : {device}")
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14').to(device)
    model = model.eval()
    print(images_folder) 
    print(type(images_folder))
    images_fit = os.listdir(images_folder)
    
    
    liste_tokens_fit=[extract_features_from_one_image(f"{images_folder}/{i}", model,device) for i in images_fit]
    tok_fit =  combine_features(liste_tokens_fit)
    fit_pca2(tok_fit)

In [8]:
fit_pca("echantillon_fit")#input the folder with the images you want to use for fit here

✅ Using CUDA GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Device utilisé : cuda


Using cache found in C:\Users\fanti/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\fanti/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\fanti/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\fanti/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


echantillon_fit
<class 'str'>
PCA_fit saved as pca_reference.pkl


In [8]:
def tranform_pca2(tok_trans): 
    '''Loads the previously fitted pca and applies it on the features extracted from an image.'''
    pca = joblib.load("pca_reference.pkl")
    if hasattr(tok_trans, 'cpu'):
        tok_trans = tok_trans.detach().cpu().numpy()
    return pca.transform(tok_trans)
    

In [9]:
def transform_pca(images_folder):
    "Applies transform_pca_2 to a folder of images. output a dictionnary with an image name for key, and PCA components as val"
    device = which_device()
    print(f"Device utilisé : {device}")
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14').to(device)
    model = model.eval()
   
    liste_images = os.listdir(images_folder)
    liste_reduced= []
    for i in liste_images:
        print(i)
        liste_reduced.append(tranform_pca2(extract_features_from_one_image(f"{images_folder}/{i}", model,device)))
    
    dico_reduced = dict(zip(liste_images, liste_reduced))
    return(dico_reduced)
    

In [13]:
dico_pim = transform_pca("echantillon_test") #input here the image folder on which you want to apply the previously fitted PCA. Then either use it to fit a 2nd PCA, or to apply this 2nd PCA if you already fitted the said PCA.

✅ Using CUDA GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Device utilisé : cuda


Using cache found in C:\Users\fanti/.cache\torch\hub\facebookresearch_dinov2_main


10848876_1.jpg
10848877_1.jpg
10848878_1.jpg
10848879_1.jpg
10848879_2.jpg
10849172_1.jpg
10849173_1.jpg
10849174_1.jpg
10849175_1.jpg
10849176_1.jpg
10849177_1.jpg
10849178_1.jpg
10849179_1.jpg
10849180_1.jpg
10849181_1.jpg
10849181_2.jpg
10849182_1.jpg
10849183_1.jpg
10849184_1.jpg
10849186_1.jpg
10849188_1.jpg
10849188_2.jpg
10849189_1.jpg
10849191_1.jpg
10849192_1.jpg
10849193_1.jpg
10849194_1.jpg
10849194_2.jpg
10849195_1.jpg
10849196_1.jpg
10849196_2.jpg
10849197_1.jpg
10849197_2.jpg
10849198_1.jpg
10849199_1.jpg
10849200_1.jpg
10849200_2.jpg
10849201_1.jpg
10849203_1.jpg
10849204_1.jpg
10849204_2.jpg
10849205_1.jpg
10849205_2.jpg
10849206_2.jpg
10849208_1.jpg
10849209_1.jpg
10849210_1.jpg


In the following sequence, we fit and execute a second PCA, but this time only on a part of the image, selected thanks to the first PCA. This allows us to better focus the PCA on a specific part of the image, and exclude the rest. I.e the PCA 1 component will very often allow us to discriminate background from foreground, and therefor allow us only on one of these two. 

In [16]:
def second_fit(images_folder, threshold=0):
    """"""
    # we use an arbitrary threshold here, but it can be changed, if lower or higher threshold create more useful discriminations.
    # First we gather the pca results
    dico_reduced = transform_pca(images_folder)
    noms_images = sorted(os.listdir(images_folder))

    patches_retenus = []

    for nom in noms_images:
        tok = dico_reduced[nom]  # (N_patches, n_components)

        tok_2d = einops.rearrange( # replaces the first dimension of tok by the actual size of the image (N_PATCH and N_PATCH), it allows us to then create the mask on the image 
            tok,
            '(r c) p -> r c p',
            r=N_PATCHES,
            c=N_PATCHES
        )

       
        masque = tok_2d[..., 0] > threshold 
        rows, cols = np.where(masque) #so here we get the coordinates of every patch where PCA1 > threshold. In practice it corresponds to parts of the images, generaly background with these parameters

        for r, c in zip(rows, cols):
            patches_retenus.append(tok_2d[r, c, :]) #and here we add the features of the patches to our list "patches retenus"

    
    patches_retenus = np.array(patches_retenus)
    print(f"Patches retenus : {len(patches_retenus)}")
#and now we fit a new PCA with only these patches, that will theoricaly now distinguish things inside the foreground of our image.
    pca = PCA(n_components=N_PCA_COMPONENTS)
    pca.fit(patches_retenus)
    joblib.dump(pca, f"pca_reference_{threshold}.pkl")



In [12]:
second_fit("echantillon_fit") #input an image folder, and a threshold.

✅ Using CUDA GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Device utilisé : cuda


Using cache found in C:\Users\fanti/.cache\torch\hub\facebookresearch_dinov2_main


10849387_1.jpg
10849391_1.jpg
10849402_1.jpg
10849404_1.jpg
10849414_1.jpg
10849415_1.jpg
10849428_1.jpg
Patches retenus : 33264


Now that we have 2 reference PCA, you need to go back in the notebook to generate a new dico_pim, that you can input in the following functions to generate (faulty) visualisations of your results.

In [14]:
def crop_on_pca_x(dico_reduced, images_folder, pca_rank, treshold_min, treshold_max, threshold_pc1):
    """This function takes as input a dictionnary of formatting {name_image : 1st PCA result}, the corresponding image folder, the pca rank on which we want to work and the value thresholds of the pixels we want to capture for that rank, eventualy threshold pca1 corresponds to the threshold we chose when fitting on pca1 previously.
    It produces an image containing the biggest identified zone of patches corresponding to the selected thresholds"""
    
  #load the pca previously fitted on pca1
    pca = joblib.load(f"pca_reference_{threshold_pc1}.pkl")
    
    
    for nom_image, pair_pca in dico_reduced.items(): #load the images and convert to array
        im1 = Image.open(os.path.join(images_folder,nom_image)).convert('RGB')
        im1 = im1.resize((IMG_SIZE, IMG_SIZE))  # same size as for fit
        im1 = np.array(im1)                     # convert to numpy for cropping
        n_patches_h = IMG_SIZE // PATCH_SIZE
       
        n_patches_w = IMG_SIZE // PATCH_SIZE
       
        pair_pca_2d = einops.rearrange(
            pair_pca,
            '(r c) p -> r c p',
            r=n_patches_h,
            c=n_patches_w
        )#same as previously we add the two dimensions to be able to project our array in two dimesions (and have a coherent mask with out image)
        pair_pcax = pair_pca_2d[..., int(pca_rank - 1)]  # (r, c)
       
        # We first exclude all the patches which value is under the threshold determined for pc1 (since our input already contains the results of a pca)
        PC1_THRESHOLD = threshold_pc1 
        good_pc1 = pair_pca_2d[..., 0] > PC1_THRESHOLD
        patches_pc1 = pca.transform(pair_pca_2d[good_pc1, :])  
        pair_pca_transformed = np.zeros((n_patches_h, n_patches_w, pca.n_components))
        pair_pca_transformed[good_pc1]= patches_pc1
        pcax_transformed= pair_pca_transformed[..., int(pca_rank -1)]
        # And now we do the thresholding operation to extract only the parts of the image that we are interested in.
        good_pcx = (pcax_transformed >= treshold_min) & (pcax_transformed <= treshold_max)

        good = good_pc1 & good_pcx

        rows, cols = np.where(good)
        
        good_pca = pca.transform(pair_pca_2d[good, :]) # useless and to delete?
        im1_orig = np.array(Image.open(f"{images_folder}\\{nom_image}").convert('RGB')) #now we load the original image to crop the corresponding parts and get a visual result
        h_orig, w_orig = im1_orig.shape[:2]
    
        masque = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

        for r, c in zip(rows, cols):
            y, x = r * PATCH_SIZE, c * PATCH_SIZE
            masque[y:y + PATCH_SIZE, x:x + PATCH_SIZE] = 255  # we put everything in white for the background, can be edited if needed
        
        n_labels, labels = cv2.connectedComponents(masque) #this cv2 function will create connected components out of our patches

        tailles = [(labels == label).sum() for label in range(1, n_labels)]
        plus_grand_label = np.argmax(tailles) + 1  # +1 because we excluded label 0

        # now we create a mask containing only the biggest (np.argmax) of the connected components
        masque_filtre = np.zeros_like(masque)
        masque_filtre[labels == plus_grand_label] = 255

# now we fill this biggest connected component
        contours, _ = cv2.findContours(masque_filtre, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        masque_rempli = np.zeros_like(masque_filtre)
        cv2.fillPoly(masque_rempli, contours, 255)

        # and apply it to the original image
        masque_orig = cv2.resize(masque_rempli, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
        masque_3d = np.stack([masque_orig] * 3, axis=-1)
        image_masquee = np.where(masque_3d > 0, im1_orig, 255)

        # and save the output in a png image, obviously the name and path can be edited
        nom_sortie = f"v2{pca_rank}_{treshold_min}_{treshold_max}_{nom_image}_masque_pca_biggest_blanc.png"
        Image.fromarray(image_masquee).save(nom_sortie)
        

    return str("fini")

In [15]:
def crop_on_pca_x_min_size(dico_reduced, images_folder, pca_rank, treshold_min, treshold_max, threshold_pc1, minimal_size):
    """exactly the same as the previous one, but instead of taking only the biggest zone, we take everyzone superior to the minimal_size variable"""
    
  #load the pca previously fitted on pca1
    pca = joblib.load(f"pca_reference_{threshold_pc1}.pkl")
    
    
    for nom_image, pair_pca in dico_reduced.items(): #load the images and convert to array
        im1 = Image.open(f"{images_folder}\\{nom_image}").convert('RGB')
        im1 = im1.resize((IMG_SIZE, IMG_SIZE))  
        im1 = np.array(im1)                     

        n_patches_h = IMG_SIZE // PATCH_SIZE
        print(n_patches_h)
        n_patches_w = IMG_SIZE // PATCH_SIZE
        print(n_patches_w)
        pair_pca_2d = einops.rearrange(
            pair_pca,
            '(r c) p -> r c p',
            r=n_patches_h,
            c=n_patches_w
        )
        pair_pcax = pair_pca_2d[..., int(pca_rank - 1)]  # (r, c)
       
     
        PC1_THRESHOLD = threshold_pc1  # same value as in second_fit
        good_pc1 = pair_pca_2d[..., 0] > PC1_THRESHOLD
        row_full, col_full = np.where(good_pc1)
        im1_orig = np.array(Image.open(f"{images_folder}\\{nom_image}").convert('RGB'))
        h_orig, w_orig = im1_orig.shape[:2]
    
        masque = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        for r, c in zip(row_full, col_full):
            y, x = r * PATCH_SIZE, c * PATCH_SIZE
            masque[y:y + PATCH_SIZE, x:x + PATCH_SIZE] = 255
        masque_filtre = np.zeros_like(masque)
        masque_orig = cv2.resize(masque, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
        masque_3d = np.stack([masque_orig] * 3, axis=-1)
        image_masquee = np.where(masque_3d > 0, im1_orig, 255)

     
        nom_sortie = f"detour_{nom_image}_blanc.png"
        Image.fromarray(image_masquee).save(nom_sortie)


        patches_pc1 = pca.transform(pair_pca_2d[good_pc1, :])  # transform, pas fit_transform
        pair_pca_transformed = np.zeros((n_patches_h, n_patches_w, pca.n_components))
        pair_pca_transformed[good_pc1]= patches_pc1
        
        pcax_transformed= pair_pca_transformed[..., int(pca_rank -1)]
        good_pcx = (pcax_transformed >= treshold_min) & (pcax_transformed <= treshold_max)

      
        good = good_pc1 & good_pcx

        rows, cols = np.where(good)
        
        im1_orig = np.array(Image.open(f"{images_folder}\\{nom_image}").convert('RGB'))
        h_orig, w_orig = im1_orig.shape[:2]
      
        masque = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

        for r, c in zip(rows, cols):
            y, x = r * PATCH_SIZE, c * PATCH_SIZE
            masque[y:y + PATCH_SIZE, x:x + PATCH_SIZE] = 255  # patch retenu = blanc
        n_labels, labels = cv2.connectedComponents(masque)

        tailles = [(labels == label).sum() for label in range(1, n_labels)]

        # Masque avec uniquement la plus grande zone
        masque_filtre = np.zeros_like(masque)
        for label in range(1, n_labels):
            zone = (labels == label)
            if zone.sum()>= minimal_size:
                masque_filtre[zone]= 255

# Remplacer le masque original
       
        masque=masque_filtre
        # Appliquer à l'image
        masque_orig = cv2.resize(masque, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
        masque_3d = np.stack([masque_orig] * 3, axis=-1)
        image_masquee = np.where(masque_3d > 0, im1_orig, 255)

        # Sauvegarder
        nom_sortie = f"{pca_rank}_{treshold_min}_{treshold_max}_{nom_image}_masque_pca_sup5000_blanc.png"
        Image.fromarray(image_masquee).save(nom_sortie)
        

    return str("fini")

In [17]:
crop_on_pca_x_min_size(dico_pim, "echantillon_test", 2, 8, 12,0, 5000)

100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100
100


'fini'